# Spatial perturbation benchmark — run
Runs the 2x2 (seed x propagation) benchmark, prints the metric table, and draws the result
figures. Run on the lab server (env `spatial-tumor-ai`) so it can read the real .h5mu.

In [ ]:
import matplotlib
%matplotlib inline
from spbench.adapters import get_adapter
from spbench.config import run_benchmark
from spbench.viz import (plot_2x2, plot_attribution, plot_learned_value,
                         plot_significance_contrast, plot_slope, plot_seed_vs_learned,
                         plot_skill_leaderboard)
import numpy as np, pandas as pd

## 1. Load data via adapter (swap adapter/path here to change dataset)

In [ ]:
DIR='/home/yiru/database/spatial_perturbed_processed/CRISPR_based/Saunders_2025_40513557'
data = get_adapter('saunders')(DIR, max_files=4).load()
print(data.n_cells, data.n_genes, 'perturbations:', len(data.perturbations()))

## 2. Define the evaluation set
`SIGNIFICANT` = the perturbations MC-spatial flagged as having real spatial effects (p<0.05). `NON_SIGNIFICANT` = a random sample of the *same size* drawn from every other perturbation (a proxy negative-control group — the vast majority have no real spatial signal). This balanced contrast checks the learned model does not just help everywhere regardless of signal.

In [ ]:
SIGNIFICANT = ['Pck1','Rrbp1','Hspd1','Psmc1','Sepp1','Bcl2l1','Vcp',
               'Ass1','Pten','Rrn3','Letm1','Hspa5','Sec61b','Rngtt']
sig = [p for p in SIGNIFICANT if p in set(data.perturbations())]
others = [p for p in data.perturbations() if p not in set(SIGNIFICANT)]
rng = np.random.default_rng(0)   # fixed seed for reproducibility
NON_SIGNIFICANT = list(rng.choice(others, size=min(len(sig), len(others)), replace=False))
EVAL = sig + NON_SIGNIFICANT
print('evaluating', len(EVAL), '=', len(sig), 'significant +',
      len(NON_SIGNIFICANT), 'random non-significant')

## 3. Run the benchmark (trivial seed + Gaussian baseline + GCN)

In [ ]:
res = run_benchmark(data, perturbations=EVAL, k=15, k_ref=5,
                    gcn_kwargs={'hidden':64,'epochs':30})

## 4. Metric table (per-perturbation 2x2 + attribution + leakage)

In [ ]:
rows=[]
for p in EVAL:
    g=res['grids'][p]; a=res['attribution'][p]
    rows.append(dict(perturbation=p, sig=p in set(SIGNIFICANT),
                     e1=g['1']['energy_prop'], e2=g['2']['energy_prop'],
                     e3=g['3']['energy_prop'], e4=g['4']['energy_prop'],
                     seed_cost=a['seed_cost'], learned_value=a['learned_value'],
                     end_to_end=a['end_to_end'], leak_ok=res['leakage_pass'][p]))
df=pd.DataFrame(rows).sort_values('end_to_end'); df

## 5. Result figures
Absolute E-distance is background-dominated (~6); the signal is in the **differences**. These figures plot the differences across perturbations and contrast significant vs non-significant — that is what reveals whether learned propagation captures real signal.

**Headline — skill leaderboard:** absolute E-distance turned into a 0..100% skill (fraction of recoverable niche signal captured) via `calibrate_edistance`. Only perturbations whose perturbed niche sits clearly above the noise floor are shown; the rest are dropped (no signal to predict). <0 = worse than predicting 'no effect'.

In [ ]:
plot_skill_leaderboard(res).savefig('fig_skill_leaderboard.png', dpi=140)

**B — the headline:** learned_value distribution, significant vs non-significant (box + points + sign-test). If the significant group is not clearly higher, the GCN's gain is generic (a better niche smoother), not signal-specific.

In [ ]:
plot_significance_contrast(res, SIGNIFICANT).savefig('fig_B_contrast.png', dpi=140)

**A — per perturbation:** learned_value sorted, coloured by significance (>0 = GCN beats Gaussian).

In [ ]:
plot_learned_value(res, SIGNIFICANT).savefig('fig_A_learned_value.png', dpi=140)

**C — consistency + outliers:** baseline (e1) -> learned (e2) slope per perturbation (downward = GCN better). Outliers sit far from the cluster.

In [ ]:
plot_slope(res, SIGNIFICANT).savefig('fig_C_slope.png', dpi=140)

**D — attribution scatter:** seed_cost vs learned_value (where does the error live?).

In [ ]:
plot_seed_vs_learned(res, SIGNIFICANT).savefig('fig_D_seed_vs_learned.png', dpi=140)

**Single-gene 2x2 (explainer, not a result):** what one perturbation's grid looks like.

In [ ]:
plot_2x2(res['grids'][EVAL[0]], title=EVAL[0])

## 6. Ranking + go/no-go note
Lowest end-to-end E-distance = best. The real go/no-go is the rho_niche (with vs without niche) +0.10 gate — wire that in once the no-niche variant exists. Note: learned_value > 0 alone does **not** prove signal-specificity; compare the two groups in figure B, and add a label-permutation control.

In [ ]:
print('ranking (best first):', res['ranking'])